# Modelling Training Notebook

This notebook trains the LSTM model on the data created in previous sections

In [ ]:
import sys
sys.path.append('../src')

In [ ]:
import os
from pprint import pprint
import pickle
import json
from datetime import datetime
import warnings
import random

import math
import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from matplotlib_venn import venn2
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    root_mean_squared_error,
    r2_score
)
from sklearn.exceptions import UndefinedMetricWarning

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate, BatchNormalization
from tensorflow.keras.metrics import RootMeanSquaredError, MeanAbsoluteError, MeanAbsolutePercentageError
from tensorflow.keras.callbacks import EarlyStopping

import mlflow
from inequalipy import gini

from util_IO import (
    load_pickle_from_main_project_dir,
    load_timeseries_df,
    load_attributes_df,
    get_stadiamaps_provider_api_key_in_env,
    get_mlflow_tracking_uri,
    geo_plot_basic
)

# Set pandas to display a maximum of 300 columns
pd.set_option('display.max_columns', 300)
pd.set_option('display.max_rows', 1000)

# Suppress the SettingWithCopyWarning
pd.options.mode.chained_assignment = None

pd.set_option('display.float_format', '{:.3f}'.format)

# Suppresses warnings related to the R² score by year
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

# Set seed
random.seed(82)
np.random.seed(82)
tf.random.set_seed(82)

# Retrieve provider for maps
stadiamaps_provider = get_stadiamaps_provider_api_key_in_env()

# Set mlflow server
mlflow.set_tracking_uri(
    uri=get_mlflow_tracking_uri()
)

# Set autolog properties
mlflow.keras.autolog()

## Configuration

In [ ]:
# Validation run?
VALIDATION = False
if VALIDATION:
    validation_run = "1" # 1, 2 or 3

# Store model?
SAVE_MODEL = False

### Load metadata from *1-DataAggregation.ipynb*

In [ ]:
aggr_parameters_dict, camels_gb_use_case_dir = load_pickle_from_main_project_dir(
    'aggr_parameters_dict.pkl'
)

# # Print imported variable
pprint(aggr_parameters_dict)

In [ ]:
# Variables picked
date_field = aggr_parameters_dict['timeseries']['date_field']
attributes_index = aggr_parameters_dict["attributes"]["attributes_index"]
camels_gb_datasets_dir = aggr_parameters_dict["camels_gb_datasets_dir"]
camels_gb_data_attributes_aggr_dir = aggr_parameters_dict["camels_gb_data_attributes_aggr_dir"]
camels_gb_data_timeseries_aggr_dir = aggr_parameters_dict['camels_gb_data_timeseries_aggr_dir']
camels_gb_bronze_dir = aggr_parameters_dict['camels_gb_bronze_dir']
camels_gb_silver_dir = aggr_parameters_dict['camels_gb_silver_dir']

temp_files_dir = f"{camels_gb_use_case_dir}/temp"

### Model feed parameters

In [ ]:
# Model feed parameters
sequence_length = 30
label_field = 'discharge_vol' # log1p_discharge_vol / discharge_vol
start_year = 1985
cs = "Y"
feed_train_size = 7

# Derive model feed ID
if VALIDATION:
    model_feed_ID = f"w{sequence_length}-{label_field}-{start_year}-cs_{cs}-train_size_{feed_train_size}-{validation_run}"
else:
    model_feed_ID = f"w{sequence_length}-{label_field}-{start_year}-cs_{cs}-train_size_{feed_train_size}"
print(model_feed_ID)

### Set MLflow experiment name

In [ ]:
# Set MLflow Experiment linked to the model feed ID
curr_experiment = mlflow.set_experiment(
    model_feed_ID
)

# Load model feed

## Feed dictionary

In [ ]:
# Read dictionary
if VALIDATION:
    path = "validation_data/"
    temp_files_dir = f"{temp_files_dir}/validation_runs/{validation_run}"
else:
    path = ""

with open(
    os.path.join(
        camels_gb_silver_dir,
        f"{path}model_feed-{model_feed_ID}.pkl"
    ),
    'rb'
) as f:
    model_feed = pickle.load(f)
    
# Print imported variable
print(model_feed.keys())

## Feed variables split

In [ ]:
for key, value in model_feed.items():
    
    globals()[key] = value
    
    try:
        if isinstance(value, list):
            print(f"`{key}` ingested with dimension: {len(value)}")
        else:
            print(f"`{key}` ingested with dimension: {value.shape}")
    except:
        print(f"`{key}` ingested")

## Model feature set

### Retrieval

In [ ]:
# Set feature set name
feature_set_name = 'gear_1' # gear_1

# Derive feaure set index for dictionary
feature_set_index = f"{feature_set_name}-cs_{cs}"

# Load dictionary (of dictionaries)
feature_sets, _ = load_pickle_from_main_project_dir(
    'feature_sets.pkl'
)

# Ingest feature set dictionary
feature_set = feature_sets[feature_set_index]

# Retrieve list for attributes
X_static_vars_names = feature_set['X_static_vars_names']

# Retrieve list for time series
X_vars_names = feature_set['X_vars_names']

# Checks
assert set(X_static_vars_names).issubset(set(X_static_cols_names))
assert set(X_vars_names).issubset(set(X_cols_names))

### Data frame variables selection

In [ ]:
# __________
# Attributes

X_train_static = X_train_static_df[X_static_vars_names].values
X_test_static = X_test_static_df[X_static_vars_names].values

# ___________
# Time series
pointers_list = [X_cols_names.index(element) for element in X_vars_names]

# Select
X_train = X_train[:,:,pointers_list]
X_test = X_test[:,:,pointers_list]

### Data sets conversion

In [ ]:
X_train = np.array(X_train, dtype=np.float32)
X_train_static = np.array(X_train_static, dtype=np.float32)

X_test = np.array(X_test, dtype=np.float32)
X_test_static = np.array(X_test_static, dtype=np.float32)

# Model train and evaluation

## Model dimensions

In [ ]:
train_val_prop = 0.8
n_obs_train, ts, n_dynamic_vars = X_train.shape
n_static_var = X_train_static.shape[1]
train_val_index = int(train_val_prop * n_obs_train)
batch_size = 128

## Model parameters

In [ ]:
LSTM_hidden_size_layer_1 = 128
LSTM_hidden_size_layer_2 = 128

encoder_dense_layer_1_multiplier = 1.5
encoder_dense_layer_2_multiplier = 0.75

concatenated_dense_layer = 64
dense_output_activation = 'exponential' # 'exponential' / 'softplus'
loss = 'MAE' # 'MAE' / 'MAPE'
early_stopping_monitor = 'val_mean_absolute_percentage_error' if loss=='MAPE' else 'val_mean_absolute_error'

## MLflow start run

In [ ]:
# Start MLflow run
run = mlflow.start_run(run_name=f"{dense_output_activation}-{loss}")

## MLflow log parameters

In [ ]:
# _____________________
# Input data parameters
mlflow.log_params(
    {
        'sequence_length': sequence_length,
        'label_field': label_field,
        'start_year': start_year,
        'cs': cs,
        'feed_train_size': train_size,
        'model_feed_ID': model_feed_ID,
        'feature_set_name': feature_set_name,
        'feature_set_index': feature_set_index,
        'train_size': train_size,
        'test_size': test_size,
        'min_required_obs': min_required_obs
    }
)

mlflow.log_params(
    {
        'X_train_dim': " | ".join([f"{num:,}" for num in X_train.shape]),
        'y_train_dim': " | ".join([f"{num:,}" for num in y_train.shape]),
        'X_test_dim': " | ".join([f"{num:,}" for num in X_test.shape]),
        'y_test_dim': " | ".join([f"{num:,}" for num in y_test.shape]),
        'X_train_static_df_dim': " | ".join([f"{num:,}" for num in X_train_static_df.shape]),
        'X_test_static_df_dim': " | ".join([f"{num:,}" for num in X_test_static_df.shape]),
        'X_train_registry_df_dim': " | ".join([f"{num:,}" for num in X_train_registry_df.shape]),
        'X_test_registry_df_dim': " | ".join([f"{num:,}" for num in X_test_registry_df.shape])
    }
)


# ________________
# Input data lists
lists_to_log_names = [
    'X_static_no_scaled_columns_names_list',
    'X_minmax_columns_names_list',
    'X_no_scaled_columns_names_list',
    'X_static_standard_columns_names_list',
    'X_static_cols_names',
    'X_standard_columns_names_list',
    'X_cols_names',
    'X_static_vars_names',
    'X_vars_names'
]

for list_name in lists_to_log_names:

    temp_file = os.path.join(
        temp_files_dir,
        f"{list_name}.json"
    )
    
    # Dump the data to a JSON file
    with open(temp_file, 'w') as file:
        json.dump(globals()[list_name], file)

    # Log as artifact
    mlflow.log_artifact(
        temp_file,
        "parameter_lists"
    )


# ________________
# Model dimensions
mlflow.log_params(
    {
        'train_val_prop': train_val_prop,
        'n_obs_train': n_obs_train,
        'ts': ts,
        'n_dynamic_vars': n_dynamic_vars,
        'n_static_var': n_static_var,
        'train_val_index': train_val_index,
        'batch_size': batch_size
    }
)


# ________________
# Model parameters
mlflow.log_params(
    {
        'LSTM_hidden_size_layer_1': LSTM_hidden_size_layer_1,
        'LSTM_hidden_size_layer_2': LSTM_hidden_size_layer_2,
        'encoder_dense_layer_1_multiplier': encoder_dense_layer_1_multiplier,
        'encoder_dense_layer_2_multiplier': encoder_dense_layer_2_multiplier,
        'concatenated_dense_layer': concatenated_dense_layer,
        'dense_output_activation': dense_output_activation,
        'loss': loss,
        'early_stopping_monitor': early_stopping_monitor
    }
)

## Model definition

In [ ]:
# ___________
# LSTM branch
lstm_input = Input(shape=(sequence_length, n_dynamic_vars), name='lstm_input')
lstm = LSTM(LSTM_hidden_size_layer_1, return_sequences=True)(lstm_input)
lstm = LSTM(LSTM_hidden_size_layer_2)(lstm)


# ______________
# Encoder branch
encoder_input = Input(shape=(n_static_var,), name='encoder_input')

encoder = (
    Dense(
        n_static_var * encoder_dense_layer_1_multiplier,
        activation='relu'
    )
)(encoder_input)


encoder = (
    Dense(
        n_static_var * encoder_dense_layer_2_multiplier,
        activation='relu'
    )
)(encoder)

# ________________________
# Concatenation and output

# Concatenate the outputs of the two branches
concatenated = Concatenate()([lstm, encoder])
concatenated = Dense(concatenated_dense_layer, activation='relu')(concatenated)
concatenated = BatchNormalization()(concatenated)

# Pass the concatenated output to a dense layer
dense_output = Dense(1, activation=dense_output_activation, name='output')(concatenated)

# Create the model
model = Model(inputs=[lstm_input, encoder_input], outputs=dense_output)

In [ ]:
# Compile the model
model.compile(
    optimizer='adam',
    loss=loss,
    metrics=[RootMeanSquaredError(), MeanAbsoluteError(), MeanAbsolutePercentageError()]
)

# Print the model summary
model.summary()

## Dataset preparation

In [ ]:
# Create the training dataset
train_dataset = (
    tf.data.Dataset
        .from_tensor_slices(
            (
                (X_train[:train_val_index], X_train_static[:train_val_index]),
                y_train[:train_val_index]
            )
        )
        .batch(batch_size)
)


# Create the validation dataset
val_dataset = (
    tf.data.Dataset
        .from_tensor_slices(
            (
                (X_train[train_val_index:], X_train_static[train_val_index:]),
                y_train[train_val_index:]
            )
        )
        .batch(batch_size)
)

## Model training

In [ ]:
# _______________
# Train the model

# Disable the autolog
mlflow.tensorflow.autolog(disable=True)

# EarlyStopping
early_stopping = EarlyStopping(
    monitor=early_stopping_monitor,
    patience=3,
    restore_best_weights=True
)


# FIT!
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    # epochs=1,
    epochs=50,
    callbacks=[early_stopping]
)

## Store the model as a pickle file

In [ ]:
# Store the model
if SAVE_MODEL:
    with open(f"../models/trained_model.pkl", 'wb') as f:
        pickle.dump(model, f)

## Train set evaluation

In [ ]:
# Evaluate the model on test data
train_loss, train_rmse, train_mae, train_mape = model.evaluate([X_train, X_train_static], y_train)
print(f"Root Mean Squared Error:\t{train_rmse}\nMean Absolute Error:\t\t{train_mae}\nMean Absolute Percentage Error:\t{train_mape}")

# Log test metrics
mlflow.log_metrics({
    'train_loss': train_loss,
    'train_rmse': train_rmse,
    'train_mae': train_mae,
    'train_mape': train_mape,
})

## Test set evaluation

In [ ]:
# Evaluate the model on test data
test_loss, test_rmse, test_mae, test_mape = model.evaluate([X_test, X_test_static], y_test)
print(f"Root Mean Squared Error:\t{test_rmse}\nMean Absolute Error:\t\t{test_mae}\nMean Absolute Percentage Error:\t{test_mape}")

# Log test metrics
mlflow.log_metrics({
    'test_loss': test_loss,
    'test_rmse': test_rmse,
    'test_mae': test_mae,
    'test_mape': test_mape,
})

## Predictions

In [ ]:
Predictions
y_train_hat = model.predict(
    train_dataset.concatenate(val_dataset)
)

y_test_hat = model.predict([X_test, X_test_static])

## Anti-transformation

In [ ]:
# Grab the prefix from the name of the field used for label
prefix = label_field.split("_")[0]

# Define the anti-transformation function
if prefix=="log1p":
    anti_transformation_func = np.expm1
else:
    anti_transformation_func = lambda x: x

print(f"Function applied:\t{anti_transformation_func}")

# Anti-transformation
y_train_anti_transf = anti_transformation_func(y_train)
y_test_anti_transf = anti_transformation_func(y_test)

y_train_hat_anti_transf = anti_transformation_func(y_train_hat).reshape(-1)
y_test_hat_anti_transf = anti_transformation_func(y_test_hat).reshape(-1)

# Error analysis

## Catchments geo-references

### Data frame creation

In [ ]:
# Load polygons
gdf_catchments = gpd.read_file(
    os.path.join(
        camels_gb_bronze_dir,
        'CAMELS_GB_catchment_boundaries',
        'CAMELS_GB_catchment_boundaries.shp'
    )
)

# Derive area in squared km
gdf_catchments['area_sqkm'] = gdf_catchments['geometry'].area / 1e6

### Retrieving flow gauging station locations

In [ ]:
# Retrieving flow gauging station locations
gauging_station_locations = load_attributes_df(
    camels_gb_data_attributes_aggr_dir,
    "fundamental_locations_postFEa.csv",
    attributes_index
)

# Convert flow gauging station locations pandas df into geopandas
gdf_gauging_station_locations = (
    gpd.GeoDataFrame(
        gauging_station_locations,
        geometry=(
            gpd.points_from_xy(
                gauging_station_locations['gauge_lon'],
                gauging_station_locations['gauge_lat']
            )
        ),
        crs="EPSG:4326"
    )
)
gdf_gauging_station_locations = gdf_gauging_station_locations.to_crs(gdf_catchments.crs)


# Merge
# ⚠️ `right` join -> `gdf_catchments` reduces in rows
gdf_catchments = (
    gdf_catchments
        .merge(
            gdf_gauging_station_locations,
            left_on='ID_STRING',
            right_index=True,
            how='right',
            suffixes=(
                '_catchment',
                '_flow-gauging-station'
            )
        )
)

assert gdf_catchments.isna().sum().sum() == 0, "Some unexpected NaN"


# Rename the geometry columns
gdf_catchments = (
    gdf_catchments
        .rename(
            columns={
                'geometry_catchment': 'catchment_geom',
                'geometry_flow-gauging-station': 'flow-gauging-station_geom'
            }
        )
)


# Set one as the active geometry
gdf_catchments = gdf_catchments.set_geometry('flow-gauging-station_geom')

### Retrieving benchmark catchments flag (Y/N)

In [ ]:
# Retrieving benchmark catchments
benchmark_catchments = load_attributes_df(
    camels_gb_data_attributes_aggr_dir,
    "full_list_of_fields.csv",
    attributes_index,
    ['benchmark_catch']
)


# Merge
gdf_catchments = (
    gdf_catchments
        .merge(
            benchmark_catchments,
            left_on='ID_STRING',
            right_index=True,
            how='left'
        )
)

assert gdf_catchments.isna().sum().sum() == 0, "Some unexpected NaN"

### Retrieving Chalk stream table

In [ ]:
# Read the file into a DataFrame
chalk_streams_df = pd.read_csv(
    os.path.join(
        camels_gb_datasets_dir,
        "chalk_streams.csv"
    ),
    dtype={attributes_index: 'str'},
    index_col=attributes_index
)

gdf_catchments = (
    gdf_catchments
        .merge(
            chalk_streams_df,
            left_on='ID_STRING',
            right_index=True,
            how='left'
        )
)

assert gdf_catchments.isna().sum().sum() == 0, "Some unexpected NaN"

In [ ]:
display(gdf_catchments.head(3))

## Observations-based

### Data frames creation

In [ ]:
# ______________
# Initialization

# Define a very small number (epsilon)
epsilon = 1e-6

# Train
train_error_analysis_df = (
    pd.DataFrame({
        'y_train': y_train_anti_transf + epsilon,
        'y_train_hat': y_train_hat_anti_transf
    })
)

# Test
test_error_analysis_df = (
    pd.DataFrame({
        'y_test': y_test_anti_transf + epsilon,
        'y_test_hat': y_test_hat_anti_transf
    })
)


# _______________________
# Differences calculation 
for df, curr_set in zip(
    [train_error_analysis_df, test_error_analysis_df],
    ['train', 'test']
):
    
    # Simple difference
    simple_difference_field_name = f"y_{curr_set}_diff"
    df[simple_difference_field_name] = df[f"y_{curr_set}_hat"] - df[f"y_{curr_set}"]
    
    # Absolute difference
    absolute_difference_field_name = f"y_{curr_set}_abs_diff"
    df[absolute_difference_field_name] = df[simple_difference_field_name].abs()
    
    # % difference
    perc_difference_field_name = f"y_{curr_set}_%_diff"
    df[perc_difference_field_name] = df[simple_difference_field_name] / df[f"y_{curr_set}"]
    
    # Absolute difference
    absolute_perc_difference_field_name = f"y_{curr_set}_abs_%_diff"
    df[absolute_perc_difference_field_name] = df[perc_difference_field_name].abs()
    

# ____________
# Add registry

# Train
train_error_analysis_df = (
    pd.concat([
            train_error_analysis_df,
            X_train_registry_df
        ],
        axis=1
    )
)

# Test
test_error_analysis_df = (
    pd.concat([
            test_error_analysis_df,
            X_test_registry_df
        ],
        axis=1
    )
)


# __________
# Add `year`

# Train
train_error_analysis_df['year'] = (
    pd.to_datetime(
        train_error_analysis_df['end_date']
    )
    .dt
    .year
)

# Test
test_error_analysis_df['year'] = (
    pd.to_datetime(
        test_error_analysis_df['end_date']
    )
    .dt
    .year
)

### Add geo locations

In [ ]:
# Define a function to merge the data
def add_geo(gdf_catchments, geo_columns, error_analysis_df):
    
    merged_df = (
        error_analysis_df
            .merge(
                gdf_catchments[geo_columns],
                left_on='catchmentID',
                right_on='ID_STRING',
                how='left'
            )
            .drop(columns=geo_columns[0])
        )
    
    assert merged_df.isna().sum().sum() == 0, "Some unexpected NaN"
    return merged_df

# Columns to insert
geo_columns = [
    'ID_STRING',
    'benchmark_catch'
]

# Merge
train_error_analysis_df = add_geo(
    gdf_catchments,
    geo_columns,
    train_error_analysis_df
)

test_error_analysis_df = add_geo(
    gdf_catchments,
    geo_columns,
    test_error_analysis_df
)

In [ ]:
display(train_error_analysis_df.head(3))
display(test_error_analysis_df.head(3))

### Add ALL SET of `postFEa` (=not scaled) variables

In [ ]:
# Load timeseries
timeseries_df = load_timeseries_df(
    camels_gb_data_timeseries_aggr_dir,
    "timeseries_postFEa.csv",
    date_field
)

# Load attributes
attributes_df = load_attributes_df(
    camels_gb_data_attributes_aggr_dir,
    "fundamental_postFEa.csv",
    attributes_index
)

display(timeseries_df.head(3))
display(attributes_df.head(3))

In [ ]:
# Define a function to merge and sort the data
def add_ts_and_attributes(ts, attributes, error_analysis_df):

    # Memorize initial status
    n_obs_before_merge = error_analysis_df.shape[0]

    # Unpack
    ts_df, ts_columns = ts
    attribute_df, attributes_columns = attributes

    # Merge time series
    merged_df = (
        error_analysis_df
            .merge(
                ts_df[ts_columns],
                left_on=['catchmentID', 'end_date'],
                right_on=['catchmentID', 'date'],
                how='left'
            )
            .drop(columns=['date'])
        )
    
    assert merged_df.isna().sum().sum() == 0, "Some unexpected NaN while merging time series"
    assert n_obs_before_merge == merged_df.shape[0], "Row size mismatch while merging time series"

    # Merge attributes
    merged_df = (
        merged_df
            .merge(
                attribute_df[attributes_columns],
                left_on='catchmentID',
                right_index=True,
                how='left'
            )
        )

    assert merged_df.isna().sum().sum() == 0, "Some unexpected NaN while merging attributes"
    assert n_obs_before_merge == merged_df.shape[0], "Row size mismatch while merging attributes"

    return merged_df


# Merge
ts_cols = [
    'precipitation',
    'temperature',
    'humidity',
    'shortwave_rad',
    'longwave_rad',
    'windspeed',
    'sin_year',
    'cos_year',
    'time_ref'
]

attr_cols = [
    'baseflow_index',
    'sand_perc',
    'silt_perc',
    'clay_perc',
    'organic_perc',
    'gauge_elev',
    'area',
    'dpsbar',
    'elev_mean',
    'elev_min',
    'elev_10',
    'elev_50',
    'elev_90',
    'elev_max',
    'dwood_perc',
    'ewood_perc',
    'grass_perc',
    'shrub_perc',
    'crop_perc',
    'urban_perc',
    'inwater_perc',
    'bares_perc',
    'surfacewater_abs',
    'groundwater_abs',
    'discharges',
    'num_reservoir',
    'reservoir_cap',
    'chalk_stream_flag'
]

# 🚩 'catchmentID' & 'date' needed to perform the merge
train_error_analysis_df = add_ts_and_attributes(
    (timeseries_df, ts_cols + ['catchmentID', 'date']),
    (attributes_df, attr_cols),
    train_error_analysis_df
)

# 🚩 'catchmentID' & 'date' needed to perform the merge
test_error_analysis_df = add_ts_and_attributes(
    (timeseries_df, ts_cols + ['catchmentID', 'date']),
    (attributes_df, attr_cols),
    test_error_analysis_df
)

display(train_error_analysis_df.head(3))
display(test_error_analysis_df.head(3))

### Log data frames

In [ ]:
# Save
train_error_analysis_df.to_csv(
    os.path.join(
        temp_files_dir,
        f"train_error_analysis_df.csv"
    )
)
    
    
test_error_analysis_df.to_csv(
    os.path.join(
        temp_files_dir,
        f"test_error_analysis_df.csv"
    )
)


# Log
for csv_file in [f"train_error_analysis_df.csv", f"test_error_analysis_df.csv"]:

    mlflow.log_artifact(
        os.path.join(
            temp_files_dir,
            csv_file
        ),
        "error_analysis/obs/tables"
    )

## Log Catchments

In [ ]:
# Extract unique catchment IDs
unique_catchments = train_error_analysis_df['catchmentID'].unique()
unique_catchments_df = pd.DataFrame(unique_catchments, columns=['catchmentID'])

# Save to CSV
unique_catchments_df.to_csv(os.path.join(temp_files_dir,'train_unique_catchment_ids.csv'), index=False)

# Extract unique catchment IDs
unique_catchments = test_error_analysis_df['catchmentID'].unique()
unique_catchments_df = pd.DataFrame(unique_catchments, columns=['catchmentID'])

# Save to CSV
unique_catchments_df.to_csv(os.path.join(temp_files_dir,'test_unique_catchment_ids.csv'), index=False)

print("Unique catchment IDs have been saved to 'train_unique_catchment_ids.csv'.")
print("Unique catchment IDs have been saved to 'test_unique_catchment_ids.csv'.")

### Calculate distribution statistics

In [ ]:
# Define stats calculation function
def detailed_describe(df):

    df = df.select_dtypes(include='number')
    desc = df.describe(percentiles=[.01, .05, .1, .15, .2, .25, .3, .5, .7, .75, .8, .85, .9, .95, .99])
    desc.loc['skew'] = df.skew()
    desc.loc['kurt'] = df.kurt()
    desc.loc['range'] = df.max() - df.min()

    return desc

# Calculate stats
dd_train_error_analysis_df = detailed_describe(train_error_analysis_df)
dd_test_error_analysis_df = detailed_describe(test_error_analysis_df)

# Display
display(dd_train_error_analysis_df)
display(dd_test_error_analysis_df)


# Save
dd_train_error_analysis_df.to_csv(
    os.path.join(
        temp_files_dir,
        "dd_train_error_analysis_df.csv"
    )
)
    
    
dd_test_error_analysis_df.to_csv(
    os.path.join(
        temp_files_dir,
        "dd_test_error_analysis_df.csv"
    )
)


# Log
for csv_file in ["dd_train_error_analysis_df.csv", "dd_test_error_analysis_df.csv"]:

    mlflow.log_artifact(
        os.path.join(
            temp_files_dir,
            csv_file
        ),
        "error_analysis/obs/tables"
    )

### Log observations metrics

In [ ]:
# Function for calculation of metrics
def calculate_metrics(
    train_df,
    test_df,
    filter_condition=None,
    suffix=''
):
    
    if (suffix and not filter_condition) or (filter_condition and not suffix):
        raise ValueError("Both filter_condition and suffix must be provided together")

    if suffix != '':
        suffix = f'_{suffix}'

    metrics = {}

    for set_name, df in [
        ('train', train_df),
        ('test', test_df)
    ]:
        if filter_condition is not None:
            df = df.query(filter_condition)
        
        y_true_col = f'y_{set_name}'
        y_pred_col = f'y_{set_name}_hat'
        
        metrics[f'obs_{set_name}_MAE{suffix}'] = mean_absolute_error(df[y_true_col], df[y_pred_col])
        metrics[f'obs_{set_name}_MAPE{suffix}'] = mean_absolute_percentage_error(df[y_true_col], df[y_pred_col])
        metrics[f'obs_{set_name}_NSE{suffix}'] = r2_score(df[y_true_col], df[y_pred_col])
        metrics[f'obs_{set_name}_RMSE{suffix}'] = root_mean_squared_error(df[y_true_col], df[y_pred_col])

    return metrics

# All set
observations_metrics = calculate_metrics(
    train_error_analysis_df, test_error_analysis_df
)

# Filter on `chalk_stream_flag`
observations_metrics.update(
    calculate_metrics(
        train_error_analysis_df,
        test_error_analysis_df,
        filter_condition='chalk_stream_flag == False',
        suffix='without_chalk'
    )
)

# Filter on `benchmark_catch`
observations_metrics.update(
    calculate_metrics(
        train_error_analysis_df,
        test_error_analysis_df,
        filter_condition='benchmark_catch == "Y"',
        suffix='benchmark_only'
    )
)

pprint({key: observations_metrics[key] for key in sorted(observations_metrics)})
mlflow.log_metrics(observations_metrics)

### Viz

#### Function definitions

In [ ]:
def scatter_errors_vs_label(
    df_list,
    ys_list,
    sets_list,
    x_min=None,
    x_max=None,
    y_min=None,
    y_max=None
):

    # List len
    n_row = len(ys_list)
    n_column = len(sets_list)

    # Create a figure with 2 subplots
    fig, axes = plt.subplots(n_row, n_column, figsize=(8*n_column, 5*n_row))

    # Ensure axes is always a 2D array
    if n_row == 1 and n_column == 1:
        axes = [[axes]]
    elif n_row == 1:
        axes = [axes]
    elif n_column == 1:
        axes = [[ax] for ax in axes]

    # Loop on column (data frame)
    for col in range(n_column):

        # Define current df
        df = df_list[col]

        # Define current set
        curr_set = sets_list[col]

        # Loop on row (variables)
        for row in range(n_row):

            # Define suffix/var
            y_suffix = ys_list[row]

            # Plot scatter plot
            sns.scatterplot(
                data=df,
                x=f"y_{curr_set}",
                y=f"y_{curr_set}_{y_suffix}",
                ax=axes[row, col]
            )
            axes[row, col].set_xlim(x_min, x_max)
            axes[row, col].set_ylim(y_min, y_max)
            axes[row, col].set_xlabel(f"y_{curr_set}")
            axes[row, col].set_ylabel(f"y_{curr_set}_{y_suffix}")

            # Add a red line for the average
            avg_train = df[f"y_{curr_set}_{y_suffix}"].mean()
            axes[row, col].axhline(avg_train, color='red', linestyle='--', linewidth=1)

    # Adjust layout and show the plot
    plt.tight_layout()
    return fig

In [ ]:
def hist_error_analysis(
    df_list,
    ys_list,
    sets_list,
    x_min=None,
    x_max=None,
    bins=50
):
    # List lengths
    n_row = len(ys_list)
    n_column = len(sets_list)

    # Create a figure with subplots
    fig, axes = plt.subplots(n_row, n_column, figsize=(8 * n_column, 5 * n_row))

    # Ensure axes is always a 2D array
    if n_row == 1 and n_column == 1:
        axes = [[axes]]
    elif n_row == 1:
        axes = [axes]
    elif n_column == 1:
        axes = [[ax] for ax in axes]

    # Loop through columns (dataframes)
    for col in range(n_column):
        df = df_list[col]
        curr_set = sets_list[col]

        # Loop through rows (variables)
        for row in range(n_row):
            y_suffix = ys_list[row]

            # Plot histogram
            sns.histplot(
                data=df,
                x=f"y_{curr_set}_{y_suffix}",
                ax=axes[row][col],
                kde=False,
                bins=bins
            )
            axes[row][col].set_xlim(x_min, x_max)
            axes[row][col].set_xlabel(f"y_{curr_set}_{y_suffix}")
            axes[row][col].set_ylabel("Frequency")

            # Add red line for the mean
            avg_val = df[f"y_{curr_set}_{y_suffix}"].mean()
            axes[row][col].axvline(avg_val, color='red', linestyle='--', linewidth=1)

    plt.tight_layout()
    return fig

In [ ]:
def hist_errors_vs_vars(
    df_list,
    x,
    ys_list,
    sets_list,
    x_min=None,
    x_max=None,
    y_min=None,
    y_max=None,
    frac=0.01
):

    # List len
    n_row = len(ys_list)
    n_column = len(sets_list)

    # Create a figure with 2 subplots
    fig, axes = plt.subplots(n_row, n_column, figsize=(8*n_column, 5*n_row))

    # Ensure axes is always a 2D array
    if n_row == 1 and n_column == 1:
        axes = [[axes]]
    elif n_row == 1:
        axes = [axes]
    elif n_column == 1:
        axes = [[ax] for ax in axes]

    # Loop on column (data frame)
    for col in range(n_column):

        # Define current df
        df = df_list[col].sample(frac=frac, random_state=82)

        # Define current set
        curr_set = sets_list[col]

        # Loop on row (variables)
        for row in range(n_row):

            # Define suffix/var
            y_suffix = ys_list[row]

            # Plot bivariate histogram
            sns.histplot(
                data=df,
                x=x,
                y=f"y_{curr_set}_{y_suffix}",
                ax=axes[row, col],
                bins=30
            )
            
            axes[row, col].set_xlim(x_min, x_max)
            axes[row, col].set_ylim(y_min, y_max)

    fig.suptitle(
        x,
        fontsize=16,
        y=0.945
    )
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    
    return fig

#### Scatter plots

##### All differences vs. label

In [ ]:
# Lists
df_list = [train_error_analysis_df, test_error_analysis_df]
ys_list = ['diff', 'abs_diff', '%_diff', 'abs_%_diff']
sets_list = ['train', 'test']

# Plot
fig = scatter_errors_vs_label(
    df_list,
    ys_list,
    sets_list
)

# Log
mlflow.log_figure(
    fig,
    "error_analysis/obs/charts/y_vs_all_diffs.png"
)

# Show
plt.show()

##### Absolute diff.(s) vs. label (zoomed)

In [ ]:
# Lists
df_list = [train_error_analysis_df, test_error_analysis_df]
ys_list = ['abs_diff', 'abs_%_diff']
sets_list = ['train', 'test']

# Focus
y_min = 0
y_max = 10

# Plot
fig = scatter_errors_vs_label(
    df_list,
    ys_list,
    sets_list,
    y_min=y_min,
    y_max=y_max
)

# Log
mlflow.log_figure(
    fig,
    f"error_analysis/obs/charts/y_vs_abs_diffs_{y_min}_{y_max}.png"
)

# Show
fig.show()

#### Residuals histograms

##### All differences

In [ ]:
# Lists
df_list = [train_error_analysis_df, test_error_analysis_df]
ys_list = ['diff', 'abs_diff', '%_diff', 'abs_%_diff']
sets_list = ['train', 'test']

# Plot
fig = hist_error_analysis(
    df_list,
    ys_list,
    sets_list,
    bins=90
)

# Log
mlflow.log_figure(
    fig,
    "error_analysis/obs/charts/hist_all_diffs.png"
)

# Show
fig.show()

#### Bivariate histograms

##### Errors vs. variables (original values)

In [ ]:
# Lists
df_list = [train_error_analysis_df, test_error_analysis_df]
ys_list = ['diff', 'abs_diff', '%_diff', 'abs_%_diff']
sets_list = ['train', 'test']

# Loop on variables
for var in (ts_cols + attr_cols):
    
    # Plot
    fig = hist_errors_vs_vars(
        df_list,
        var,
        ys_list,
        sets_list,
    )
    
    # Log
    mlflow.log_figure(
        fig,
        f"error_analysis/obs/charts/errors_vs_vars/errors_vs_{var}.png"
    )
    
    # Show
    plt.show()

## Catchments-based

### Aggregation functions definitions

In [ ]:
def get_combined_agg_funcs(set_name, df):

    # Function to calculate NSE
    def calculate_nse(group):
        return r2_score(group[f"y_{set_name}"], group[f"y_{set_name}_hat"])
    
    # Define the percentiles to calculate
    # 🚩 The way to name percentiles is set to be aligned with..
    #..general practice in the field 🚩  
    percentiles = {
        'percentile_95': 0.05,
        'percentile_70': 0.30,
        'percentile_10': 0.90,
        'percentile_5': 0.95
    }
    
    # Create the aggregation functions dynamically for both "y" and "y_hat"
    agg_funcs = {}
    for col in [f"y_{set_name}", f"y_{set_name}_hat"]:
        agg_funcs.update({
            f'{col}_median': (col, 'median'),
            f'{col}_avg': (col, 'mean'),
            **{f'{col}_{name}': (col, lambda x, q=q: x.quantile(q)) for name, q in percentiles.items()}
        })
    
    # Add additional aggregations
    additional_agg_funcs = {
        f"y_{set_name}_diff_avg": (f"y_{set_name}_diff", 'mean'),
        f"y_{set_name}_abs_diff_avg": (f"y_{set_name}_abs_diff", 'mean'),
        f"y_{set_name}_%_diff_avg": (f"y_{set_name}_%_diff", 'mean'),
        f"y_{set_name}_abs_%_diff_avg": (f"y_{set_name}_abs_%_diff", 'mean'),
        f"y_{set_name}_nse": ('catchmentID', lambda x: calculate_nse(df.loc[x.index])),
        'n_windows': (f"y_{set_name}", 'count')
    }
    
    # Combine agg_funcs and additional_agg_funcs
    combined_agg_funcs = {**additional_agg_funcs, **agg_funcs}

    return combined_agg_funcs

In [ ]:
def get_catchments_grouped_by(
    group_by_lists,
    df,
    set_name
):

    # Define output
    results = []

    # Set aggregation function to use
    combined_agg_funcs = get_combined_agg_funcs(
        set_name,
        df
    )

    # Loop on group-by(s)
    for group_by_list in group_by_lists:

        # Initialize specific data frame
        curr_grouped_by_df = (
            df.groupby(
                group_by_list
            )
            .agg(**combined_agg_funcs)
            .sort_values(
                f"y_{set_name}_abs_%_diff_avg",
                ascending=True
            )
        )

        # Store the data frame
        results.append(curr_grouped_by_df)

    return tuple(results)

### Data frames creation

In [ ]:
# Define group-by list of lists
group_by_lists = [
    ['catchmentID'],
    ['catchmentID', 'year']
]


# _______________________________
# Calculate grouped-by data frame

# Train
train_catchment_error_df, train_catchment_year_error_df = (
    get_catchments_grouped_by(
        group_by_lists,
        train_error_analysis_df,
        'train'
    )
)

# Test
test_catchment_error_df, test_catchment_year_error_df = (
    get_catchments_grouped_by(
        group_by_lists,
        test_error_analysis_df,
        'test'
    )
)


# _____________________________________________
# Calculate cumulated and normalized indicators
for df, set_name in [(train_catchment_error_df, 'train'), (test_catchment_error_df, 'test')]:    

    # Cumulated
    df[f"y_{set_name}_abs_%_diff_avg_cum"] = df[f"y_{set_name}_abs_%_diff_avg"].cumsum()
    df['n_windows_cum'] = df['n_windows'].cumsum()
    df['sample_percentage'] = (
        np.arange(1, df.shape[0] + 1) / 
            df.shape[0] 
                * 100
    )

    # Normalized the cumulative error per catchment
    df['mean_cum_percentage'] = (
        df[f"y_{set_name}_abs_%_diff_avg_cum"] / 
            df[f"y_{set_name}_abs_%_diff_avg_cum"].iloc[-1] # Max value
                * 100
    )

    # Normalize the cumulative number of windows per catchment
    df['n_windows_cum_percentage'] = (
        df['n_windows_cum'] / 
            df['n_windows_cum'].iloc[-1] # Max value
                * 100
    )

# ____
# Save
train_catchment_error_df.to_csv(
    os.path.join(
        temp_files_dir,
        "train_catchment_error_df.csv"
    )
)

train_catchment_year_error_df.to_csv(
    os.path.join(
        temp_files_dir,
        "train_catchment_year_error_df.csv"
    )
)

test_catchment_error_df.to_csv(
    os.path.join(
        temp_files_dir,
        "test_catchment_error_df.csv"
    )
)

test_catchment_year_error_df.to_csv(
    os.path.join(
        temp_files_dir,
        "test_catchment_year_error_df.csv"
    )
)


# ____
# Log

for csv_file in [
    "train_catchment_error_df.csv",
    "train_catchment_year_error_df.csv",
    "test_catchment_error_df.csv",
    "test_catchment_year_error_df.csv"
]:

    mlflow.log_artifact(
        os.path.join(
            temp_files_dir,
            csv_file
        ),
        "error_analysis/catchment/tables"
    )

In [ ]:
display(train_catchment_error_df.head(3))
display(test_catchment_error_df.head(3))

### Log catchments metrics

In [ ]:
catchments_metrics = {
    'catch_train_mean:avg_MAE': train_catchment_error_df['y_train_abs_diff_avg'].mean(),
    'catch_train_mean:avg_MAPE': train_catchment_error_df['y_train_abs_%_diff_avg'].mean(),
    'catch_train_mean:avg_NSE': train_catchment_error_df['y_train_nse'].mean(),

    'catch_test_mean:avg_MAE': test_catchment_error_df['y_test_abs_diff_avg'].mean(),
    'catch_test_mean:avg_MAPE': test_catchment_error_df['y_test_abs_%_diff_avg'].mean(),
    'catch_test_mean:avg_NSE': test_catchment_error_df['y_test_nse'].mean(),
}

pprint(catchments_metrics)

mlflow.log_metrics(catchments_metrics)

### Problematic catchment(s)

#### Gini's index on errors

In [ ]:
y_train_abs_diff_avg_gini = gini(train_catchment_error_df['y_train_abs_diff_avg'].values)
y_test_abs_diff_avg_gini = gini(test_catchment_error_df['y_test_abs_diff_avg'].values)

y_train_abs_perc_diff_avg_gini = gini(train_catchment_error_df['y_train_abs_%_diff_avg'].values)
y_test_abs_perc_diff_avg_gini = gini(test_catchment_error_df['y_test_abs_%_diff_avg'].values)


print("Gini's index for train absolute diff. average for catchments:\t" +
      f"{y_train_abs_diff_avg_gini:.4f}")
print("Gini's index for test absolute diff. average for catchments:\t" +
      f"{y_test_abs_diff_avg_gini:.4f}")

print("Gini's index for train absolute % diff. average for catchments:\t" +
      f"{y_train_abs_perc_diff_avg_gini:.4f}")
print("Gini's index for test absolute % diff. average for catchments:\t" +
      f"{y_test_abs_perc_diff_avg_gini:.4f}")


# Log
mlflow.log_metrics(
    {
        'catch_train_gini:avg_MAE': y_train_abs_diff_avg_gini,
        'catch_test_gini:avg_MAE': y_test_abs_diff_avg_gini,
        'catch_train_gini:avg_MAPE': y_train_abs_perc_diff_avg_gini,
        'catch_test_gini:avg_MAPE': y_test_abs_perc_diff_avg_gini,
    }
)

#### Identification

In [ ]:
def identify_right_tail_catchments(df):
    
    # Difference in the cumulated normalized error line
    delta_mean_cum_percentage = df['mean_cum_percentage'].diff()

    # Retrieve list of catchments where the slope is >= 45 degrees
    right_tail_catchments = (
        df[delta_mean_cum_percentage >= 1 / (df.shape[0] - 1)]
        .index
        .to_list()
    )

    return right_tail_catchments


# ________________________________
# Calculate right tail catchments
train_right_tail_catchments = identify_right_tail_catchments(train_catchment_error_df)
test_right_tail_catchments = identify_right_tail_catchments(test_catchment_error_df)


# ________________
# Derive the quote
train_right_tail_catchments_quote = (
    len(train_right_tail_catchments) / train_catchment_error_df.shape[0]
)
test_right_tail_catchments_quote = (
    len(test_right_tail_catchments) / test_catchment_error_df.shape[0]
)


# ________________________________
# Add boolean column on data frame
train_catchment_error_df['RightTailCatchments'] = (
    train_catchment_error_df.index.isin(train_right_tail_catchments)
)
test_catchment_error_df['RightTailCatchments'] = (
    test_catchment_error_df.index.isin(test_right_tail_catchments)
)


# ___________________________________________________
# Create dataframe with right tail catchments details
train_right_tail_catchments_df = (
    gauging_station_locations
        .loc[train_right_tail_catchments]
)

test_right_tail_catchments_df = (
    gauging_station_locations
        .loc[test_right_tail_catchments]
)


# ____
# Save
train_right_tail_catchments_df.to_csv(
    os.path.join(
        temp_files_dir,
        "train_right_tail_catchments_df.csv"
    )
)
    
test_right_tail_catchments_df.to_csv(
    os.path.join(
        temp_files_dir,
        "test_right_tail_catchments_df.csv"
    )
)

# ___
# Log
for csv_file in ["train_right_tail_catchments_df.csv", "test_right_tail_catchments_df.csv"]:

    mlflow.log_artifact(
        os.path.join(
            temp_files_dir,
            csv_file
        ),
        "error_analysis/catchment/tables"
    )

### Viz

#### NSE

In [ ]:
# Create a figure with 1 row and 2 columns, sharing the y-axis
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Data and titles for the subplots
datasets = [
    (train_catchment_error_df, 'train'),
    (test_catchment_error_df, 'test')
]

# Plot NSE density for both datasets
for ax, (data, set_name) in zip(axes, datasets):
    sns.histplot(
        data=data,
        x=f"y_{set_name}_nse",
        color='skyblue',
        edgecolor='blue',
        ax=ax
    )
    ax.set_title(f"{set_name.capitalize()}")
    ax.set_xlabel(f"NSE")

# Adjust layout
plt.tight_layout()

# Log
mlflow.log_figure(
    fig,
    "error_analysis/catchment/charts/nse.png"
)

# Show
plt.show()

In [ ]:
# Create a figure with 1 row and 2 columns
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Data and titles for the subplots
datasets = [
    (train_catchment_error_df, 'train'),
    (test_catchment_error_df, 'test')
]

# Plot R² for both datasets
for ax, (data, set_name) in zip(axes, datasets):
    sns.histplot(
        data=data,
        x=f"y_{set_name}_nse",
        hue='RightTailCatchments',
        ax=ax,
        kde=False
    )
    ax.set_title(f"{set_name.capitalize()}")
    ax.set_xlabel(f"NSE")

# Adjust layout
plt.tight_layout()

# Log
mlflow.log_figure(
    fig,
    "error_analysis/catchment/charts/nse_plob_split.png"
)

# Show
plt.show()

#### Venn graphs

In [ ]:
# Example sets
train_right_tail_catchments_set = set(train_right_tail_catchments)
test_right_tail_catchments_set = set(test_right_tail_catchments)

# Create a Venn diagram for two sets
fig = plt.figure(figsize=(10, 5))
venn2(
    [train_right_tail_catchments_set, test_right_tail_catchments_set],
    ('Train', 'Test')
)

# Log
mlflow.log_figure(
    fig,
    "error_analysis/catchment/charts/venn_graph_train-test.png"
)

# Show
plt.show()

#### Errors concentration

In [ ]:
# Create a figure with 2 rows and 2 columns
fig, axes = plt.subplots(2, 2, figsize=(20, 15))

# Data and titles for the subplots
datasets = [
    (train_catchment_error_df, train_error_analysis_df, 'Train'),
    (test_catchment_error_df, test_error_analysis_df, 'Test')
]

# Plot cumulative error curves and box plots for both datasets
for col, (catchment_df, error_df, set_name) in enumerate(datasets):
    # Plot the cumulative error curve in the first subplot
    axes[0, col].plot(
        catchment_df['sample_percentage'], catchment_df['mean_cum_percentage'],
        marker='o', linestyle='-', label='Cumulative error'
    )
    axes[0, col].plot(
        catchment_df['sample_percentage'], catchment_df['n_windows_cum_percentage'],
        marker='o', linestyle='-', color='orange', label='Cumulative n. windows'
    )
    axes[0, col].plot(
        [0, 100], [0, 100], linestyle='--', color='grey', label='45° line'
    )
    axes[0, col].set_title(f"{set_name} - Cumulative Error")
    axes[0, col].set_xlabel('Catchments (%)')
    axes[0, col].set_ylabel('Catchment Normalized Avg. Cumulative Absolute Percentage Error\n(%)')
    axes[0, col].legend()
    axes[0, col].grid(which='both', linestyle='--', linewidth=0.5)
    axes[0, col].minorticks_on()
    # Set axis limits to exactly 0 to 100
    axes[0, col].set_xlim(0, 100)
    axes[0, col].set_ylim(0, 100)
    # Set more granular x-axis labels
    axes[0, col].set_xticks(np.arange(0, 101, 5))
    # Remove the top and right spines
    axes[0, col].spines['top'].set_visible(False)
    axes[0, col].spines['right'].set_visible(False)

    # Plot the box plot in the second subplot
    sns.boxplot(
        x='catchmentID', y=f"y_{set_name.lower()}_abs_%_diff",
        order=catchment_df.index, data=error_df, ax=axes[1, col]
    )
    axes[1, col].set_title(f"{set_name} - Box-plot of abs_%_diff by catchment")
    axes[1, col].set_xlabel('CatchmentID')
    axes[1, col].set_ylabel('abs % diff')
    # Remove the top and right spines
    axes[1, col].spines['top'].set_visible(False)
    axes[1, col].spines['right'].set_visible(False)
    # Remove x-axis ticks and label
    axes[1, col].set_xticks([])
    axes[1, col].set_xlabel('')

# Adjust layout and show the plot
plt.subplots_adjust(hspace=0.5)
plt.tight_layout()

# Log
mlflow.log_figure(
    fig,
    "error_analysis/catchment/charts/errors_concentration.png"
)

# Show
plt.show()

#### Add geo locations

In [ ]:
# Define a function to merge and sort the data
def merge_and_sort(gdf_catchments, catchment_error_df, set_name):
    columns = [
        f'y_{set_name.lower()}_diff_avg',
        f'y_{set_name.lower()}_abs_diff_avg',
        f'y_{set_name.lower()}_%_diff_avg',
        f'y_{set_name.lower()}_abs_%_diff_avg',
        f'y_{set_name.lower()}_nse',
    ]
    merged_df = (
        gdf_catchments
            .merge(
                catchment_error_df[columns],
                left_on='ID_STRING',
                right_index=True,
                how='right'
            )
            .sort_values(columns, ascending=False)
    )
    assert merged_df.isna().sum().sum() == 0, "Some unexpected NaN"
    return merged_df

# Merge and sort for train set
train_gdf_catchments_merged = merge_and_sort(gdf_catchments, train_catchment_error_df, 'Train')

# Merge and sort for test set
test_gdf_catchments_merged = merge_and_sort(gdf_catchments, test_catchment_error_df, 'Test')

# Display the results
display(train_gdf_catchments_merged.head(3))
display(test_gdf_catchments_merged.head(3))

##### Area

In [ ]:
# Set x_min and x_max
x_min = 0
x_max = 200

# Create a figure with 2 rows and 2 columns
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)

# Data and titles for the subplots
datasets = [
    (train_gdf_catchments_merged, 'y_train_abs_%_diff_avg', 'Train'),
    (test_gdf_catchments_merged, 'y_test_abs_%_diff_avg', 'Test')
]

# Plot scatter plots without limits in the first row
for ax, (data, y_column, title) in zip(axes[0], datasets):
    sns.scatterplot(data=data, x='area_sqkm', y=y_column, ax=ax)
    ax.set_xlabel("Catchment Area (sq km)")
    ax.set_ylabel("abs_%_diff")
    ax.set_title(f'{title}')

# Plot scatter plots with limits in the second row
for ax, (data, y_column, title) in zip(axes[1], datasets):
    sns.scatterplot(data=data, x='area_sqkm', y=y_column, ax=ax)
    ax.set_xlim(x_min, x_max)
    ax.set_xlabel(f"Catchment Area (sq km) - focus on range [{x_min} , {x_max}]")
    ax.set_ylabel("abs_%_diff")

# Adjust layout
plt.tight_layout()

# Log
mlflow.log_figure(
    fig,
    "error_analysis/catchment/charts/area_vs_abs_diffs_small_&_area_focus.png"
)

# Show
plt.show()

##### Error plotted on map

In [ ]:
def get_norm(metric, vmin, vmax):

    if metric in ['diff_avg', '%_diff_avg', 'nse']:
        norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    else:
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    return norm


def format_colorbar_ticks(
    colorbar,
    lb_powerlimits=-3,
    ub_powerlimits=3
):
    colorbar.formatter.set_powerlimits((lb_powerlimits, ub_powerlimits))
    colorbar.formatter.set_useMathText(True)


def get_error_map(
    train_gdf,
    test_gdf,
    metric_name,
    cmap,
    stadiamaps_provider,
    filter_train=None,
    filter_test=None
):
    
    # Define column names based on the pattern
    y_train_col = f'y_train_{metric_name}'
    y_test_col = f'y_test_{metric_name}'

    # Get colour map object
    cmap = plt.get_cmap(cmap)
    
    # Apply the colormap to the error metric columns for train and test data
    train_color = train_gdf[y_train_col].apply(lambda x: mcolors.to_hex(cmap(x)))
    test_color = test_gdf[y_test_col].apply(lambda x: mcolors.to_hex(cmap(x)))

    # Create a figure with 2 rows and 1 column
    fig, axes = plt.subplots(1, 2, figsize=(10, 20))

    # Apply filters if provided
    if filter_train is not None:
        train_gdf_queried = train_gdf.query(filter_train)
        train_color = train_color.loc[train_gdf_queried.index]
    else:
        train_gdf_queried = train_gdf
    if filter_test is not None:
        test_gdf_queried = test_gdf.query(filter_test)
        test_color = test_color.loc[test_gdf_queried.index]
    else:
        test_gdf_queried = test_gdf

    # Plot for train data
    geo_plot_basic(train_gdf_queried, stadiamaps_provider, ax=axes[0], color=train_color)
    axes[0].set_title('Train')
    axes[0].set_xlabel(metric_name)

    # Plot for test data
    geo_plot_basic(test_gdf_queried, stadiamaps_provider, ax=axes[1], color=test_color)
    axes[1].set_title('Test')
    axes[1].set_xlabel(metric_name)

    # Create ScalarMappable for train data
    norm_train = get_norm(metric_name, train_gdf[y_train_col].min(), train_gdf[y_train_col].max())
    sm_train = plt.cm.ScalarMappable(cmap=cmap, norm=norm_train)
    sm_train.set_array([])
    # sm_train.set_clim(train_gdf_queried[y_train_col].min(), train_gdf_queried[y_train_col].max())

    # Create ScalarMappable for test data
    norm_test = get_norm(metric_name, test_gdf[y_test_col].min(), test_gdf[y_test_col].max())
    sm_test = plt.cm.ScalarMappable(cmap=cmap, norm=norm_test)
    sm_test.set_array([])
    # sm_test.set_clim(test_gdf_queried[y_test_col].min(), test_gdf_queried[y_test_col].max())
    

    # Add the colorbar to the plot as a legend for train data
    cbar_train = fig.colorbar(sm_train, ax=axes[0], orientation='horizontal', fraction=0.01, pad=0.015)
    # cbar_train.ax.set_ylim(
    #     train_gdf_queried[y_train_col].min(),
    #     train_gdf_queried[y_train_col].max()
    # )

    # Add the colorbar to the plot as a legend for test data
    cbar_test = fig.colorbar(sm_test, ax=axes[1], orientation='horizontal', fraction=0.01, pad=0.015)
    # cbar_test.ax.set_ylim(
    #     test_gdf_queried[y_test_col].min(),
    #     test_gdf_queried[y_test_col].max()
    # )

    # Adjust ticks
    format_colorbar_ticks(cbar_train)
    format_colorbar_ticks(cbar_test)

    # Adjust layout and show the plot
    plt.tight_layout()
    
    return fig

In [ ]:
# Set variables and cmap
vars_plotted_on_map = [
    ('diff_avg', 'coolwarm_r'),
    ('abs_diff_avg', 'Reds'),
    ('%_diff_avg', 'coolwarm_r'),
    ('abs_%_diff_avg', 'Reds'),
    ('nse', 'RdYlGn')
]

In [ ]:
# ALL CATCHMENTS IN SAMPLE

for geometry in ['catchment_geom', 'flow-gauging-station_geom']:

    # Set geometry
    (
        train_gdf_catchments_merged
            .set_geometry(
                geometry,
                inplace=True
            )
    )
    
    (
        test_gdf_catchments_merged
            .set_geometry(
                geometry,
                inplace=True
            )
    )

    for var, cmap in vars_plotted_on_map:
        
        # Get plot
        fig = get_error_map(
            train_gdf_catchments_merged,
            test_gdf_catchments_merged,
            var,
            cmap,
            stadiamaps_provider,
        )
    
        # Log
        mlflow.log_figure(
            fig,
            f"error_analysis/catchment/charts/errors_map_{var}_{geometry}_all_sample.png",
            save_kwargs={'bbox_inches': 'tight'}
        )
    
        # Show
        plt.show()

In [ ]:
# BENCHMARK CATCHMENTS (in sample)

for geometry in ['catchment_geom', 'flow-gauging-station_geom']:

    # Set geometry
    (
        train_gdf_catchments_merged
            .set_geometry(
                geometry,
                inplace=True
            )
    )
    
    (
        test_gdf_catchments_merged
            .set_geometry(
                geometry,
                inplace=True
            )
    )
    
    for var, cmap in vars_plotted_on_map:
        
        # Get plot
        fig = get_error_map(
            train_gdf_catchments_merged,
            test_gdf_catchments_merged,
            var,
            cmap,
            stadiamaps_provider,
            filter_train='benchmark_catch == "Y"',
            filter_test='benchmark_catch == "Y"'
        )
    
        # Log
        mlflow.log_figure(
            fig,
            f"error_analysis/catchment/charts/errors_map_{var}_{geometry}_uk_benchmark.png"
        )
    
        # Show
        plt.show()

In [ ]:
# CHALK CATCHMENTS (excluded)

if cs == "Y":

    for geometry in ['catchment_geom', 'flow-gauging-station_geom']:
    
        # Set geometry
        (
            train_gdf_catchments_merged
                .set_geometry(
                    geometry,
                    inplace=True
                )
        )
        
        (
            test_gdf_catchments_merged
                .set_geometry(
                    geometry,
                    inplace=True
                )
        )
        
        for var, cmap in vars_plotted_on_map:
            
            # Get plot
            fig = get_error_map(
                train_gdf_catchments_merged,
                test_gdf_catchments_merged,
                var,
                cmap,
                stadiamaps_provider,
                filter_train='chalk_stream_flag == False',
                filter_test='chalk_stream_flag == False'
            )
        
            # Log
            mlflow.log_figure(
                fig,
                f"error_analysis/catchment/charts/errors_map_{var}_{geometry}_chalk_streams_excluded.png"
            )
        
            # Show
            plt.show()

In [ ]:
# CHALK CATCHMENTS (only)

if cs == "Y":

    for geometry in ['catchment_geom', 'flow-gauging-station_geom']:
    
        # Set geometry
        (
            train_gdf_catchments_merged
                .set_geometry(
                    geometry,
                    inplace=True
                )
        )
        
        (
            test_gdf_catchments_merged
                .set_geometry(
                    geometry,
                    inplace=True
                )
        )
        
        for var, cmap in vars_plotted_on_map:
            
            # Get plot
            fig = get_error_map(
                train_gdf_catchments_merged,
                test_gdf_catchments_merged,
                var,
                cmap,
                stadiamaps_provider,
                filter_train='chalk_stream_flag == True',
                filter_test='chalk_stream_flag == True'
            )
        
            # Log
            mlflow.log_figure(
                fig,
                f"error_analysis/catchment/charts/errors_map_{var}_{geometry}_chalk_streams_only.png"
            )
        
            # Show
            plt.show()

## Time series

### Viz

#### Actual Vs. Predicted **time windows** comparison

In [ ]:
def plot_time_series(
    catchment_error_df,
    error_analysis_df,
    set_name,
    num_subplots,
    percent_right_batch,
    percent_left_batch,
    percent_medium_batch,
    save_dir=None
):
    
    np.random.seed(82)
    
    # Calculate the number of catchments for each batch
    total_catchments = len(catchment_error_df)
    num_right_batch = math.ceil(total_catchments * percent_right_batch)
    num_left_batch = math.ceil(total_catchments * percent_left_batch)
    num_medium_batch = math.ceil(total_catchments * percent_medium_batch)
    
    # Select catchments for each batch
    right_batch_catchments = catchment_error_df.index[-num_right_batch:]
    left_batch_catchments = catchment_error_df.index[:num_left_batch]
    remaining_catchments = catchment_error_df.index[num_left_batch:-num_right_batch]
    medium_batch_catchments = (
        np.random.choice(
            remaining_catchments,
            num_medium_batch,
            replace=False
        )
    )
    
    # Combine all selected catchments
    selected_catchments = np.concatenate([
        right_batch_catchments,
        left_batch_catchments,
        medium_batch_catchments
    ])
    
    # Plot time series for each catchment and group combination
    for catchment in selected_catchments:
        
        curr_nse = catchment_error_df.loc[catchment][f"y_{set_name}_nse"]
        
        # Retrieve data frame with only current catchments
        catchment_df = (
            error_analysis_df[
                error_analysis_df['catchmentID'] == catchment
            ]
        )
        
        # Derive groups
        groups = sorted(catchment_df['group'].unique())
        
        # Derive number of plots
        num_plots = math.ceil(len(groups) / num_subplots)
        
        # Looping on plots
        for i in range(num_plots):
            start_idx = i * num_subplots
            end_idx = (i + 1) * num_subplots
            groups_chunk = groups[start_idx:end_idx]
            len_groups_chunk = len(groups_chunk)
              
            # Define plot
            fig, axes = plt.subplots(
                len_groups_chunk,
                1,
                figsize=(10, len_groups_chunk * 4)
            )
            if len_groups_chunk == 1:
                axes = [axes]
            
            batch_type = (
                'right' if catchment in right_batch_catchments else
                'left' if catchment in left_batch_catchments else
                'medium'
            )
            fig.suptitle(
                f'{catchment} ({batch_type}: NSE={curr_nse:.2f}) / {str(i+1).zfill(2)} ',
                fontsize=16,
                y=0.945
            )
            
            # Define sub-plots
            for j, group in enumerate(groups_chunk):
                group_df = catchment_df[catchment_df['group'] == group]
                
                # Actual
                sns.lineplot(
                    data=group_df,
                    x='end_date',
                    y=f'y_{set_name}',
                    ax=axes[j],
                    label='Actual',
                    color='blue'
                )
                
                # Predicted
                sns.lineplot(
                    data=group_df,
                    x='end_date',
                    y=f'y_{set_name}_hat',
                    ax=axes[j],
                    label='Predicted',
                    color='red'
                )
                
                # Formatting
                axes[j].set_ylabel('')
                axes[j].set_xlabel('')
                axes[j].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m/%y'))
                axes[j].legend()
                axes[j].grid(True)

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])

            # Save the plot if save_dir is provided
            if save_dir:
                mlflow.log_figure(
                    fig,
                    os.path.join(
                        save_dir,
                        f"TS_{batch_type}_{catchment}_{str(i+1).zfill(2)}.png"
                    )
                )
                plt.close(fig)
            else:
                plt.show()

In [ ]:
def get_quotes(q):
    percent_right_batch = q
    percent_left_batch = min(
        percent_right_batch,
        1 - percent_right_batch
    )
    percent_medium_batch = min(
        1 - percent_right_batch - percent_left_batch,
        q * 2
    )

    return percent_right_batch, percent_left_batch, percent_medium_batch


# _____
# Train

# Get quotes
percent_right_batch, percent_left_batch, percent_medium_batch = get_quotes(
    train_right_tail_catchments_quote
)
print(f"Right batch quote: {percent_right_batch:.1%} - Left batch quote: {percent_left_batch:.1%} - Medium batch quote: {percent_medium_batch:.1%}")

# Get plots
plot_time_series(
    train_catchment_error_df,
    train_error_analysis_df,
    'train',
    num_subplots=5,
    percent_right_batch=percent_right_batch,
    percent_left_batch=percent_left_batch,
    percent_medium_batch=percent_medium_batch,
    save_dir="error_analysis/timeseries/actual_vs_predicted/train"
)


# _____
# Test

# Get quotes
percent_right_batch, percent_left_batch, percent_medium_batch = get_quotes(
    test_right_tail_catchments_quote
)
print(f"Right batch quote: {percent_right_batch:.1%} - Left batch quote: {percent_left_batch:.1%} - Medium batch quote: {percent_medium_batch:.1%}")

# Get plots
plot_time_series(
    test_catchment_error_df,
    test_error_analysis_df,
    'test',
    num_subplots=5,
    percent_right_batch=percent_right_batch,
    percent_left_batch=percent_left_batch,
    percent_medium_batch=percent_medium_batch,
    save_dir="error_analysis/timeseries/actual_vs_predicted/test"
)

#### Actual Vs. Predicted catchment's **flow duration curves** comparison

In [ ]:
def plot_fdc(
    error_analysis_df,
    set_name,
    save_dir=None
):

    # Define variables from `set_name`
    actual_field = f"y_{set_name}"
    predicted_field = f"y_{set_name}_hat"
    
    # Define ad-hoc data frame (already grouped by `catchmentID`)
    flow_duration_curve_grouped_df = (
        error_analysis_df[[
            'catchmentID',
            actual_field,
            predicted_field
        ]]
        .groupby('catchmentID')
    )
    
    # Loop on catchments
    for curr_catchmentID, curr_catchment_df in flow_duration_curve_grouped_df:
        
        # Pop prediction series
        y_hat_series = curr_catchment_df.pop(predicted_field)
        
        # Order by actual values
        (
            curr_catchment_df
                .sort_values(
                    ascending=False,
                    by=actual_field,
                    inplace=True
                )
        )
        
        # Reset index of current df
        curr_catchment_df.reset_index(
            drop=True,
            inplace=True
        )
        
        # Rescale index
        curr_catchment_df.index = (
            curr_catchment_df.index / 
                (curr_catchment_df.shape[0]-1)
                * 100
        )
        
        # Create a figure and axes
        fig, ax = plt.subplots(figsize=(10, 6))
    
        # Plot y_train series
        sns.lineplot(
            data=curr_catchment_df,
            x=curr_catchment_df.index,
            y=actual_field,
            ax=ax,
            label='Actual'
        )
    
        # Plot sorted y_hat_series
        sns.lineplot(
            x=curr_catchment_df.index,
            y=sorted(y_hat_series, reverse=True),
            ax=ax,
            label='Predicted'
        )
    
        # Customize the plot
        plt.title(f"{curr_catchmentID}")
        plt.xlabel('Percentage time equalled or exceeded (%)')
        plt.ylabel(r'Flow ($\mathrm{m^3/s}$)')
        
        # Set custom x-ticks
        custom_ticks = [1, 5, 10, 50, 70, 95, 99]
        plt.xticks(custom_ticks)
        
        plt.grid(
            True,
            linestyle='--',
            linewidth=0.3
        )
        
        # Save the plot if save_dir is provided
        if save_dir:
            mlflow.log_figure(
                fig,
                os.path.join(
                    save_dir,
                    f"FDC_{curr_catchmentID}.png"
                )
            )
                        
            plt.close(fig)
        else:
            plt.show()

In [ ]:
# Get plots
plot_fdc(
    train_error_analysis_df,
    'train',
    save_dir="error_analysis/timeseries/fdc/train"
)

plot_fdc(
    test_error_analysis_df,
    'test',
    save_dir="error_analysis/timeseries/fdc/test"
)

# Close run

In [ ]:
mlflow.end_run()